# GAIN - Polydispersity - EMD Vol.02 (Friendly): nanoparticle size analysis from Thermo Fisher `.emd` files

The analysis of *GAIN - Polydispersity - EMD Vol.02* (round particles **and** cubic crystals), driven from a **three-step panel** instead of variables in the code.

**How to use**: run all cells (*Run > Run All Cells*); the panel opens in the last cell.
1. **Data**: choose a file or a folder and the **particle shape** (round particles / cubic crystals). The panel reads the calibration and the detector from the metadata and tells you whether the images look usable for counting.
2. **Settings**: change the detection parameters on the left and see the result on one image on the right, updated as you go (a reduced copy of the image is used for speed). Choose which size measure the statistics use.
3. **Run**: choose what to save, check the summary and start. You get the statistics table and the pooled histogram; on disk, overlay, histogram and CSV for every image plus a log of the parameters.

**Round particles**: Otsu threshold -> optional watershed split of touching particles -> border / overlap / oversize filters -> statistics. Measures: equivalent diameter, Feret max / min, Martin diameter, aspect ratio.
**Cubic crystals** (HAADF / dark-field only): finds the thin dark lines between touching crystals, ignores dark holes and bright spots inside a crystal, and measures the **side** of every cube. Merged or stacked cubes cannot be separated: they are shown in orange and not counted. Images that are not usable (atomic resolution, too low resolution) are reported and not analysed.

A `.emd` can hold several signals (DPC files hold a dozen): the one called `HAADF` is used, otherwise the largest image.

**Dependencies**: `hyperspy`, `opencv-python-headless`, `numpy`, `scipy`, `scikit-image`, `matplotlib`, `pandas`, `ipywidgets`, `ipyfilechooser` (all in the HyperSpy bundle).


In [1]:
import os
import glob

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
from skimage.filters import gaussian, sato, sobel, apply_hysteresis_threshold
from skimage.morphology import remove_small_objects, disk
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt
import hyperspy.api as hs
import datetime
import ipywidgets as W
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

%matplotlib inline

## 1. Loading `.emd` files and metadata

`load_emd_image()` reads the pixel calibration (nm/px) and the detector from the file's metadata and normalizes the raw detector counts to 8-bit (the raw intensities are kept for the cube pipeline). `read_emd_header()` reads only the metadata, to describe a file quickly in the panel.


In [2]:
UNIT_TO_NM = {
    "nm": 1.0,
    "um": 1000.0,
    "\u00b5m": 1000.0,
    "pm": 0.001,
    "a": 0.1,
    "\u00e5": 0.1,
    "angstrom": 0.1,
    "m": 1e9,
}


def normalize_to_uint8(data, low_percentile=1.0, high_percentile=99.0):
    """
    Rescales an arbitrary-range array (e.g. raw STEM detector counts) to
    0-255 uint8, clipping the extreme percentiles first so a few outlier
    pixels don't compress the contrast of the rest of the image.
    """
    lo, hi = np.percentile(data, [low_percentile, high_percentile])
    if hi <= lo:
        lo, hi = float(data.min()), float(data.max())
    clipped = np.clip(data, lo, hi)
    scaled = (clipped - lo) / (hi - lo + 1e-12) * 255.0
    return scaled.astype(np.uint8)


def classify_darkfield(detector_name):
    """
    Guesses bright-field vs dark-field from a Thermo Fisher detector name
    (e.g. 'HAADF', 'DF-S' -> dark-field; 'BF-S' -> bright-field). Defaults
    to bright-field (False) if the name is missing or unrecognized.
    """
    if not detector_name:
        return False
    return "DF" in str(detector_name).upper()


def load_emd_image(path, signal_index=None):
    """
    Loads a Thermo Fisher Velox .emd file with HyperSpy and returns
    (gray_uint8, nm_per_px, is_darkfield, info). Calibration and
    acquisition mode are read from the file's metadata -- no scale bar
    detection needed. A .emd can contain several signals (e.g. small
    auxiliary images next to the main HAADF image): with signal_index=None
    the signal called "HAADF" is used if there is one (DPC files hold a dozen
    signals: DFS segments, A-C, B-D, iDPC, dDPC, HAADF...), otherwise the 2D
    signal with the most pixels; pass an integer to force one.
    """
    result = hs.load(path)
    signals = result if isinstance(result, list) else [result]
    if signal_index is None:
        titles = [str(s_.metadata.General.title).strip().lower() for s_ in signals]
        if "haadf" in titles:
            signal_index = titles.index("haadf")
    if signal_index is None:
        sizes = [s_.data.size if s_.axes_manager.signal_dimension == 2 else 0 for s_ in signals]
        signal_index = int(np.argmax(sizes))
    if len(signals) > 1:
        print("Note: " + str(len(signals)) + " signals found in '" + os.path.basename(path) +
              "', using #" + str(signal_index) + " '" + str(signals[signal_index].metadata.General.title) +
              "' " + str(signals[signal_index].data.shape))
    s = signals[signal_index]

    ax = s.axes_manager.signal_axes[0]
    unit_factor = UNIT_TO_NM.get(str(ax.units).lower(), 1.0)
    nm_per_px = float(ax.scale) * unit_factor

    om = s.original_metadata.as_dictionary()
    detector_name = om.get("BinaryResult", {}).get("Detector") or s.metadata.General.title
    is_darkfield = classify_darkfield(detector_name)

    gray = normalize_to_uint8(s.data)

    info = {
        "detector": detector_name,
        "raw_dtype": str(s.data.dtype),
        "raw_min": float(s.data.min()),
        "raw_max": float(s.data.max()),
        "raw": s.data.astype(np.float32),  # un-normalized intensities, used by the overlap filter
    }
    return gray, nm_per_px, is_darkfield, info


def read_emd_header(path):
    """Reads only the metadata of a .emd file (no pixel data): signal used, nm_per_px, size, fov_nm, detector, is_darkfield."""
    result = hs.load(path, lazy=True)
    signals = result if isinstance(result, list) else [result]
    titles = [str(s_.metadata.General.title).strip().lower() for s_ in signals]
    if "haadf" in titles:
        idx = titles.index("haadf")
    else:
        idx = int(np.argmax([s_.data.size if s_.axes_manager.signal_dimension == 2 else 0 for s_ in signals]))
    s = signals[idx]
    ax = s.axes_manager.signal_axes[0]
    nm_per_px = float(ax.scale) * UNIT_TO_NM.get(str(ax.units).lower(), 1.0)
    shape = s.axes_manager.signal_shape      # (width, height)
    detector = s.original_metadata.as_dictionary().get("BinaryResult", {}).get("Detector") or s.metadata.General.title
    return {"nm_per_px": nm_per_px, "width": int(shape[0]), "height": int(shape[1]), "fov_nm": nm_per_px * min(shape),
            "detector": detector, "signal": str(s.metadata.General.title), "n_signals": len(signals),
            "is_darkfield": classify_darkfield(detector)}


def list_emd_files(folder):
    return sorted(glob.glob(os.path.join(folder, "*.emd")))

## 2. Segmentation, measurement, statistics, figures

The part that does the work (the round-particle and cube pipelines of *EMD Vol.02*). `analyze_image(path, cfg)` takes a dictionary of settings (the panel builds it), converts the lengths in nm to pixels with each file's own calibration and never crashes on zero detected particles.


In [3]:
def segment_and_split_particles(gray_img, min_peak_distance=15, bright_particles=False, blur_sigma=1.1, separate_touching=True):
    """
    Segments particles on a uniform background and splits touching
    particles using watershed based on the distance transform.

    Parameters
    ----------
    gray_img : np.ndarray (2D, uint8)
    min_peak_distance : int
        Minimum distance (in pixels) between local maxima of the distance
        transform, used as markers for the watershed. Should be chosen
        based on the approximate particle radius (smaller if particles are
        small/closely packed).
    bright_particles : bool
        False for bright-field-style images where particles are darker
        than the background. True for dark-field/HAADF STEM images, where
        particles are the brightest pixels (Z-contrast).
    blur_sigma : float
        Gaussian smoothing (in pixels) applied before thresholding. Values
        up to 1.1 keep the original light smoothing; larger values (used at
        high magnification, where particles are hundreds of pixels wide and
        the noisy edges would otherwise give very low circularity) also
        enlarge the morphological clean-up kernel.

    Returns
    -------
    labels_ws : np.ndarray (2D, int) - watershed labels (0 = background)
    mask : np.ndarray (2D, bool) - binary mask of the particles
    """
    if blur_sigma <= 1.1:
        blur = cv2.GaussianBlur(gray_img, (5, 5), 0)
        kernel = np.ones((3, 3), np.uint8)
    else:
        blur = cv2.GaussianBlur(gray_img, (0, 0), sigmaX=blur_sigma)
        k = int(2 * round(blur_sigma) + 1)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

    thresh_type = cv2.THRESH_BINARY if bright_particles else cv2.THRESH_BINARY_INV
    _, th = cv2.threshold(blur, 0, 255, thresh_type + cv2.THRESH_OTSU)

    th_clean = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)
    th_clean = cv2.morphologyEx(th_clean, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = th_clean > 0
    if not separate_touching:
        labels, _ = ndi.label(mask)
        return labels, mask

    dist = ndi.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=int(max(1, min_peak_distance)), labels=mask,
                             exclude_border=False)
    markers_mask = np.zeros(dist.shape, dtype=bool)
    markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)

    labels_ws = watershed(-dist, markers, mask=mask)
    return labels_ws, mask

In [4]:
MIN_DIAM_PX_FLOOR = 6  # never accept regions smaller than this, however small min_diam_nm is (pure noise)


def measure_particles(labels_ws, nm_per_px, min_diam_px=25, min_circularity=0.5, intensity_img=None, with_martin=True):
    """
    Computes area, equivalent diameter, circularity and (if intensity_img is
    given) mean intensity for each watershed region, and filters out regions
    that are too small or not circular enough (typically noise fragments or
    edge artifacts).

    Returns
    -------
    list of dict, one per accepted particle, with keys:
        label, area_px, eq_diam_px, eq_diam_nm, circularity, solidity,
        mean_intensity (None without intensity_img), centroid, bbox
    """
    props = regionprops(labels_ws, intensity_image=intensity_img)
    particles = []
    for p in props:
        area = p.area
        perim = p.perimeter if p.perimeter > 0 else 1
        circularity = 4 * np.pi * area / (perim ** 2)
        eq_diam_px = 2 * np.sqrt(area / np.pi)

        if eq_diam_px < min_diam_px:
            continue
        if circularity < min_circularity:
            continue

        fmax, fmin = feret_diameters(p.coords)
        r0, c0 = p.bbox[0], p.bbox[1]
        cnts, _ = cv2.findContours(np.pad(p.image.astype(np.uint8), 1), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contour = max(cnts, key=cv2.contourArea) - 1 + np.array([[c0, r0]])
        particles.append({
            "label": p.label,
            "area_px": area,
            "eq_diam_px": eq_diam_px,
            "eq_diam_nm": eq_diam_px * nm_per_px,
            "circularity": circularity,
            "solidity": p.solidity,
            "mean_intensity": float(p.mean_intensity) if intensity_img is not None else None,
            "centroid": p.centroid,  # (row, col)
            "bbox": p.bbox,  # (min_row, min_col, max_row, max_col)
            "feret_max_nm": fmax * nm_per_px, "feret_min_nm": fmin * nm_per_px, "aspect_ratio": fmax / max(fmin, 1e-6),
            "martin_nm": (martin_diameter(p.image) * nm_per_px) if with_martin else float("nan"),
            "contour": contour,
        })
    return particles


def split_overlapping(particles, max_intensity_ratio):
    """
    Splits particles into (kept, excluded) by mean intensity. In dark-field
    (HAADF) images the signal adds up where particles overlap, so overlapped
    particles are brighter than isolated ones. A particle is excluded when its
    mean intensity is above max_intensity_ratio times the median particle
    intensity of the image (e.g. 1.3 = more than 30% brighter than typical).
    """
    intensities = np.array([p["mean_intensity"] for p in particles])
    limit = max_intensity_ratio * np.median(intensities)
    kept = [p for p, v in zip(particles, intensities) if v <= limit]
    excluded = [p for p, v in zip(particles, intensities) if v > limit]
    return kept, excluded

In [5]:
def polydispersity_stats(diam_nm):
    """
    Computes polydispersity statistics from a non-empty array of
    diameters (in nm). Only ever called with at least one particle --
    analyze_emd_image() handles the zero-particle case separately.

    PDI here is defined as (sigma/mean)^2, the same convention typically
    used in DLS. It is a dimensionless metric: lower values indicate a
    more monodisperse sample (e.g. PDI < 0.05 - 0.1 in TEM).
    """
    diam_nm = np.asarray(diam_nm)
    mean_d = diam_nm.mean()
    std_d = diam_nm.std(ddof=1) if diam_nm.size > 1 else 0.0
    return {
        "n": len(diam_nm),
        "mean_nm": mean_d,
        "median_nm": float(np.median(diam_nm)),
        "std_nm": std_d,
        "cv_percent": std_d / mean_d * 100,
        "pdi": (std_d / mean_d) ** 2,
        "min_nm": float(diam_nm.min()),
        "max_nm": float(diam_nm.max()),
    }

In [6]:
def open_region(mask, radius, max_radius=8):
    """
    Morphological opening of a boolean mask with a disc of `radius` pixels. For big radii it is computed on a
    reduced copy (so it stays fast) and brought back to full resolution inside the original mask.
    """
    f = max(int(np.ceil(radius / max_radius)), 1)
    pad = radius + 2
    big = np.pad(mask, pad)
    if f == 1:
        out = ndi.binary_opening(big, structure=disk(radius))
    else:
        small = cv2.resize(big.astype(np.uint8), (big.shape[1] // f, big.shape[0] // f), interpolation=cv2.INTER_AREA) > 0
        small = ndi.binary_opening(small, structure=disk(max(int(round(radius / f)), 1)))
        up = cv2.resize(small.astype(np.uint8), (big.shape[1], big.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
        out = ndi.binary_dilation(up, iterations=f) & big
    return out[pad:-pad, pad:-pad]


def segment_cubes(raw, nm_per_px, smooth_nm=0.15, ridge_sigmas_nm=(0.25, 0.4, 0.6), ridge_low_pct=92.0,
                  ridge_high_pct=98.0, min_side_nm=4.0, edge_fraction=0.5, halo_nm=0.8, grow_nm=1.5,
                  min_contrast=4.0, max_gradient=0.06, open_fraction=0.10):
    """
    Segments crystals (cubes) in a dark-field (HAADF) image, including crystals that touch each other (steps 1-3 of the
    section description). Works on the raw intensities (not the 8-bit image) and on lengths in nm.
    Returns (labels, foreground, lines, background_level); labels is all zeros if nothing as big as a crystal is found.
    """
    img = raw.astype(np.float32)
    g = gaussian(img, max(smooth_nm / nm_per_px, 1.0))

    bg = float(np.percentile(g, 2))
    low = g[g < 1.4 * bg]
    noise = 1.4826 * float(np.median(np.abs(low - np.median(low)))) if low.size > 100 else 0.05 * bg
    threshold = max(bg + 8.0 * noise, 1.5 * bg)
    fg = ndi.binary_fill_holes(g > threshold)
    min_area = max(int((min_side_nm / nm_per_px) ** 2), 16)
    fg = remove_small_objects(fg, min_area)
    if not fg.any():
        # nothing as big as a crystal (e.g. atomic resolution: the lattice breaks the foreground into specks)
        return np.zeros(fg.shape, np.int32), fg, np.zeros_like(fg), bg

    sigmas = [max(s / nm_per_px, 1.0) for s in ridge_sigmas_nm]
    lg = np.log(np.maximum(g, bg))
    ridge = sato(lg, sigmas=sigmas, black_ridges=True)
    vals = ridge[fg]
    lines = apply_hysteresis_threshold(ridge, np.percentile(vals, ridge_low_pct), np.percentile(vals, ridge_high_pct)) & fg
    lines = ndi.binary_dilation(lines, iterations=1)

    cut = fg & ~lines
    # an opening wider than the fringe left outside the outer dark line detaches it from the crystals it touches
    # where that line is interrupted (the crystals are grown back to their size afterwards)
    open_r = int(round(max(0.8 / nm_per_px, 2.0)))
    open_r = min(open_r, max(int(0.15 * min_side_nm / nm_per_px) + 2, 2) + 6)
    yy, xx = np.ogrid[-open_r:open_r + 1, -open_r:open_r + 1]
    cut = ndi.binary_opening(cut, structure=(xx ** 2 + yy ** 2) <= open_r ** 2)
    cut = remove_small_objects(cut, max(min_area // 3, 9))
    cores, n_cores = ndi.label(cut)
    # discard thin pieces (slivers along the cluster edge, specks): a crystal must be able to contain a disc of
    # about half its minimum side
    min_inradius = max(0.5 * min_side_nm / nm_per_px, 2.0)
    if n_cores:
        edt_cut = ndi.distance_transform_edt(cut)
        reach = np.asarray(ndi.maximum(edt_cut, cores, np.arange(1, n_cores + 1)))
        keep = np.zeros(n_cores + 1, bool)
        keep[1:] = reach >= min_inradius
        cores = np.where(keep[cores], cores, 0)
    # grow each core back over the gap and the halo, but no further: the fringe outside the outer dark line
    # (far from every core) must not be given to a crystal
    reach_px = max(grow_nm / nm_per_px, 3.0)
    near_core = ndi.distance_transform_edt(cores == 0) <= reach_px
    grown = watershed(ridge, cores, mask=fg & near_core)

    # slope of the log-intensity: flat for a crystal, steep for the glow around a cluster
    slope = sobel(np.log(np.maximum(gaussian(img, 1.0), 1.0)))
    dist_bg = ndi.distance_transform_edt(fg)
    halo_px = max(halo_nm / nm_per_px, 2.0)
    out = np.zeros(grown.shape, np.int32)
    k = 0
    for p in regionprops(grown):
        r0, c0, r1, c1 = p.bbox
        sl = (slice(max(r0 - 2, 0), r1 + 2), slice(max(c0 - 2, 0), c1 + 2))
        m = grown[sl] == p.label
        inner = m & (dist_bg[sl] > halo_px)
        plateau = float(np.median(g[sl][inner])) if inner.sum() >= 20 else float(np.median(g[sl][m]))
        edge_level = bg + edge_fraction * (plateau - bg)
        m2 = inner | (m & (g[sl] > edge_level))
        m2 = ndi.binary_fill_holes(m2)
        # bright blobs sitting on the rim (other phases / defects) and thin attachments are not part of the crystal:
        # an opening wider than they are removes them and costs almost nothing at the corners
        r_open = int(round(open_fraction * np.sqrt(m2.sum())))
        if r_open >= 1:
            m2 = open_region(m2, r_open)
        edt_m2 = ndi.distance_transform_edt(m2)
        core_px = edt_m2 >= 0.5 * edt_m2.max()    # central part of the region, away from its own edges
        if plateau < min_contrast * bg or float(np.median(slope[sl][core_px])) > max_gradient:
            continue   # not a crystal: the glow around a cluster, or a faint smooth feature
        lab, n = ndi.label(m2)
        if n == 0:
            continue
        if n > 1:
            sizes = ndi.sum(m2, lab, range(1, n + 1))
            m2 = lab == (1 + int(np.argmax(sizes)))
        if m2.sum() < min_area:
            continue
        k += 1
        view = out[sl]
        view[m2 & (view == 0)] = k
    return out, fg, lines, bg


def measure_cubes(labels, nm_per_px, min_rectangularity=0.80, max_aspect=1.5, min_side_px=10, exclude_border=True):
    """
    One row per crystal: side (mean of the two sides of the minimum bounding rectangle), side from the area, the two
    rectangle sides, aspect ratio, rectangularity (area / rectangle area). `valid` = counted as one cube (not cut by
    the border if exclude_border, at least min_side_px pixels, rectangular enough, not too elongated). A few pixels
    are always "rectangular" and their side is known to within the pixel size only, hence the limit in pixels.
    """
    rows = []
    h, w = labels.shape
    for p in regionprops(labels):
        r0, c0, r1, c1 = p.bbox
        mask = np.pad((labels[p.slice] == p.label).astype(np.uint8), 1)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        c = max(cnts, key=cv2.contourArea)
        (cx, cy), (rw, rh), ang = cv2.minAreaRect(c)
        long_, short_ = max(rw, rh), min(rw, rh)
        rows.append({
            "label": p.label,
            "side_nm": 0.5 * (short_ + long_) * nm_per_px,
            "side_sqrtA_nm": float(np.sqrt(p.area)) * nm_per_px,
            "rect_short_nm": short_ * nm_per_px, "rect_long_nm": long_ * nm_per_px,
            "aspect": long_ / max(short_, 1e-6),
            "rectangularity": p.area / max(rw * rh, 1e-6),
            "area_nm2": p.area * nm_per_px ** 2,
            "touches_border": r0 == 0 or c0 == 0 or r1 >= h or c1 >= w,
            "centroid": p.centroid,  # (row, col)
            "box": cv2.boxPoints(((cx + c0 - 1, cy + r0 - 1), (rw, rh), ang)),
        })
    for r in rows:
        r["too_small"] = r["rect_short_nm"] < min_side_px * nm_per_px
        r["valid"] = (not (exclude_border and r["touches_border"]) and not r["too_small"]
                      and r["rectangularity"] >= min_rectangularity and r["aspect"] <= max_aspect)
    return rows


def assess_cube_image(rows, nm_per_px, shape, min_valid=3, max_object_fraction=0.33, min_side_px=10):
    """
    Decides from the calibration (nm/px, field of view) and from what was found whether the image can be used for
    counting. Returns (usable, message). Not usable: no crystal found; the crystals fill the field of view (atomic
    resolution, too much magnification); the crystals are only a few pixels wide (resolution too low).
    With fewer than min_valid cubes the image is used, with a warning.
    """
    fov = min(shape) * nm_per_px
    ok = [r for r in rows if r["valid"]]
    if not ok:
        return False, "no valid crystal found (field of view " + format(fov, ".0f") + " nm, " + format(nm_per_px, ".3f") + " nm/px)"
    side = float(np.median([r["side_nm"] for r in ok]))
    if side > max_object_fraction * fov:
        return False, ("crystals fill the field of view (side " + format(side, ".0f") + " nm, field " + format(fov, ".0f") +
                       " nm): too much magnification")
    if side < min_side_px * nm_per_px:
        return False, "crystals too small in pixels (" + format(side / nm_per_px, ".1f") + " px): resolution too low"
    note = ""
    if len(ok) < min_valid:
        note = " - WARNING: only " + str(len(ok)) + " valid crystal(s), the statistics are weak"
    return True, ("ok (field " + format(fov, ".0f") + " nm, " + format(nm_per_px, ".3f") + " nm/px, side " +
                  format(side, ".1f") + " nm = " + format(side / nm_per_px, ".0f") + " px)" + note)

In [7]:
SPHERE_MEASURES = {
    "eq_diam_nm": ("Equivalent diameter", "nm"),
    "feret_max_nm": ("Feret diameter, maximum", "nm"),
    "feret_min_nm": ("Feret diameter, minimum", "nm"),
    "martin_nm": ("Martin diameter (mean)", "nm"),
    "aspect_ratio": ("Aspect ratio (Feret max / min)", ""),
}
CUBE_MEASURES = {
    "side_nm": ("Cube side, rectangle mean", "nm"),
    "side_sqrtA_nm": ("Cube side from area (sqrt A)", "nm"),
    "rect_short_nm": ("Short side of rectangle", "nm"),
    "rect_long_nm": ("Long side of rectangle", "nm"),
    "aspect": ("Aspect ratio (long / short)", ""),
}
MEASURES = {**SPHERE_MEASURES, **CUBE_MEASURES}
DEFAULT_MEASURE = {"sphere": "eq_diam_nm", "cube": "side_nm"}
CONTRAST_OVERRIDE = {"auto": None, "dark": False, "bright": True}


def measure_label(key):
    name, unit = MEASURES[key]
    return name + (" (" + unit + ")" if unit else "")


_DIRS = np.stack([np.cos(np.deg2rad(np.arange(0, 180, 1.0))), np.sin(np.deg2rad(np.arange(0, 180, 1.0)))], axis=1)


def feret_diameters(coords_rc):
    """(Feret max, Feret min) in pixels from the pixel coordinates (row, col) of a region: caliper widths of its convex hull."""
    pts = np.ascontiguousarray(coords_rc[:, ::-1].astype(np.float32))
    hull = cv2.convexHull(pts)[:, 0, :]
    widths = (hull @ _DIRS.T).max(axis=0) - (hull @ _DIRS.T).min(axis=0) + 1.0
    return float(widths.max()), float(widths.min())


def martin_diameter(mask, step=2.0):
    """
    Martin diameter in pixels, averaged over the directions: length of the chord that splits the area of the particle in two
    equal halves. The mask is rotated (bilinear), so the result does not depend on the pixel grid.
    """
    m = np.pad(mask.astype(np.float32), 2)
    h, w = m.shape
    n = int(np.ceil(np.hypot(h, w))) + 4
    canvas = np.zeros((n, n), np.float32)
    y0, x0 = (n - h) // 2, (n - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = m
    c = (n / 2.0, n / 2.0)
    chords = []
    for ang in np.arange(0, 180, step):
        rot = cv2.warpAffine(canvas, cv2.getRotationMatrix2D(c, float(ang), 1.0), (n, n), flags=cv2.INTER_LINEAR)
        rows = rot.sum(axis=1)
        cum = np.cumsum(rows)
        i = int(np.searchsorted(cum, cum[-1] / 2.0))
        chords.append(rows[max(i - 1, 0):i + 2].mean())
    return float(np.mean(chords))


def touches_border(p, shape):
    """True if the particle's bounding box touches the image edge (the particle is cut by the frame)."""
    min_row, min_col, max_row, max_col = p["bbox"]
    return min_row == 0 or min_col == 0 or max_row >= shape[0] or max_col >= shape[1]


def draw_overlay(color_img, particles, excluded=None, border=None, oversized=None, style="circles"):
    """Kept: red (circle with the equivalent diameter, or the true contour). Excluded as overlapping: orange. Cut by the border: cyan. Oversized: magenta."""
    overlay = color_img.copy()
    thick = max(2, int(round(min(overlay.shape[:2]) / 700)))

    def draw(group, color, dot=False):
        for p in group:
            if style == "contours":
                cv2.drawContours(overlay, [p["contour"]], -1, color, thick)
            else:
                cy, cx = p["centroid"]
                cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(p["eq_diam_px"] / 2)), color, thick)
            if dot:
                cy, cx = p["centroid"]
                cv2.circle(overlay, (int(round(cx)), int(round(cy))), 1, (0, 255, 0), thick)

    draw(oversized or [], (255, 0, 255))
    draw(border or [], (255, 255, 0))
    draw(excluded or [], (0, 165, 255))
    draw(particles, (0, 0, 255), dot=True)
    return overlay


def draw_cube_overlay(color_img, labels, rows):
    """Red contour = crystal, green rectangle = cube counted, orange = excluded (not one cube / too small), cyan = cut by the border."""
    overlay = color_img.copy()
    overlay[find_boundaries(labels, mode="inner")] = (0, 0, 255)
    line = max(2, int(round(min(overlay.shape[:2]) / 700)))
    for r in rows:
        col = (0, 255, 0) if r["valid"] else ((255, 255, 0) if r["touches_border"] else (0, 165, 255))
        cv2.polylines(overlay, [np.round(r["box"]).astype(np.int32)], True, col, line)
    return overlay


def plot_histogram(values, stats, measure, out_path=None, material="nanoparticles", show=True, bins=30):
    plt.figure(figsize=(7, 5))
    plt.hist(values, bins=bins, color="#4C72B0", edgecolor="white")
    unit = MEASURES[measure][1]
    plt.axvline(stats["mean_nm"], color="crimson", linestyle="--", linewidth=1.5,
                label="Mean = " + format(stats["mean_nm"], ".2f") + (" " + unit if unit else ""))
    plt.xlabel(measure_label(measure))
    plt.ylabel("Count")
    plt.title("Size distribution " + material + " (N=" + str(stats["n"]) + ")\nStd dev = " + format(stats["std_nm"], ".2f") + (" " + unit if unit else "") +
              ", CV = " + format(stats["cv_percent"], ".1f") + "%, PDI = " + format(stats["pdi"], ".4f"))
    plt.legend()
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close()


def _empty_result(path, gray, nm_per_px, is_darkfield, info, mask=None):
    return {"path": path, "nm_per_px": nm_per_px, "fov_nm": nm_per_px * min(gray.shape), "is_darkfield": is_darkfield,
            "detector": info["detector"], "info": info, "gray": gray, "mask": mask if mask is not None else np.zeros(gray.shape, bool),
            "particles": [], "excluded": [], "border": [], "oversized": [], "stats": None, "overlay": None, "warnings": [], "message": "", "usable": True}


def analyze_image(path, cfg, loaded=None, with_martin=True):
    """
    Full pipeline on one .emd file with the settings in `cfg` (round particles or cubes, cfg["shape"]). Returns a dict: path,
    nm_per_px, fov_nm, is_darkfield, detector, gray, mask, particles (counted), excluded (overlapping / not a single cube),
    border (cut by the image edge), oversized, stats (of the chosen measure, None if nothing counted), overlay, warnings.
    `loaded` = a pre-loaded (maybe downscaled) (gray, nm_per_px, is_darkfield, info) for the live preview.
    """
    gray, nm_per_px, is_darkfield, info = load_emd_image(path) if loaded is None else loaded
    raw = info["raw"]
    if CONTRAST_OVERRIDE[cfg["contrast"]] is not None:
        is_darkfield = CONTRAST_OVERRIDE[cfg["contrast"]]
    res = _empty_result(path, gray, nm_per_px, is_darkfield, info)
    color_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    measure = cfg["measure"]
    if cfg["shape"] == "cube":
        return _analyze_cubes(res, cfg, gray, raw, nm_per_px, is_darkfield, color_img, measure)

    blur_sigma = max(cfg["smooth_nm"] / nm_per_px, 1.1)
    min_peak = max(cfg["min_peak_distance_nm"] / nm_per_px, 3)
    labels_ws, mask = segment_and_split_particles(gray, min_peak_distance=min_peak, bright_particles=is_darkfield, blur_sigma=blur_sigma,
                                                  separate_touching=cfg["separate_touching"])
    min_diam_px = max(cfg["min_diam_nm"] / nm_per_px, MIN_DIAM_PX_FLOOR)
    particles = measure_particles(labels_ws, nm_per_px, min_diam_px=min_diam_px, min_circularity=cfg["min_circularity"],
                                  intensity_img=raw, with_martin=with_martin)
    border = []
    if cfg["exclude_border"] and particles:
        border = [p for p in particles if touches_border(p, gray.shape)]
        particles = [p for p in particles if not touches_border(p, gray.shape)]
    excluded = []
    if cfg["max_intensity_ratio"] and is_darkfield and particles:
        particles, excluded = split_overlapping(particles, cfg["max_intensity_ratio"])
    oversized = []
    if cfg["max_diameter_ratio"] and particles:
        limit = cfg["max_diameter_ratio"] * np.mean([p["eq_diam_nm"] for p in particles])
        oversized = [p for p in particles if p["eq_diam_nm"] > limit]
        particles = [p for p in particles if p["eq_diam_nm"] <= limit]
    res.update(mask=mask, particles=particles, border=border, excluded=excluded, oversized=oversized)
    warnings = res["warnings"]
    if particles:
        res["stats"] = polydispersity_stats([p[measure] for p in particles])
        med = float(np.median([p["eq_diam_nm"] for p in particles]))
        if med > res["fov_nm"] / 3:
            warnings.append("the particles fill more than a third of the field of view (median diameter " + format(med, ".0f") +
                            " nm, field " + format(res["fov_nm"], ".0f") + " nm): too much magnification for counting")
        if len(particles) < 3:
            warnings.append("only " + str(len(particles)) + " particle(s): statistics are weak")
    else:
        warnings.append("no particles detected: check the particle contrast (Data), the minimum diameter and the circularity (Settings)")
    if particles or border or excluded or oversized:
        res["overlay"] = draw_overlay(color_img, particles, excluded=excluded, border=border, oversized=oversized, style=cfg["overlay_style"])
    return res


def _analyze_cubes(res, cfg, gray, raw, nm_per_px, is_darkfield, color_img, measure):
    if not is_darkfield:
        res.update(usable=False, message="cube mode needs a dark-field (HAADF) image, this one looks like bright-field (set the particle contrast to 'Bright particles' to force it)")
        res["warnings"].append(res["message"])
        return res
    labels, fg, lines, bg = segment_cubes(raw, nm_per_px, min_side_nm=cfg["min_diam_nm"])
    rows = measure_cubes(labels, nm_per_px, min_rectangularity=cfg["cube_min_rectangularity"], max_aspect=cfg["cube_max_aspect"],
                         min_side_px=cfg["cube_min_side_px"], exclude_border=cfg["exclude_border"])
    usable, message = assess_cube_image(rows, nm_per_px, gray.shape, min_side_px=cfg["cube_min_side_px"])
    res.update(mask=fg, usable=usable, message=message)
    if rows:
        res["overlay"] = draw_cube_overlay(color_img, labels, rows)
    if usable:
        res["particles"] = [r for r in rows if r["valid"]]
        res["stats"] = polydispersity_stats([r[measure] for r in res["particles"]])
        if "WARNING" in message:
            res["warnings"].append(message.split(" - WARNING: ")[-1])
    else:
        res["warnings"].append(message)
    res["border"] = [r for r in rows if not r["valid"] and r["touches_border"] and cfg["exclude_border"]]
    res["excluded"] = [r for r in rows if not r["valid"] and r not in res["border"]]
    if not usable:
        res["particles"], res["stats"] = [], None
    return res


def particles_table(result):
    """One row per particle counted, with every measure (used for the CSV)."""
    if result.get("particles") and "side_nm" in result["particles"][0]:
        cols = ["side_nm", "side_sqrtA_nm", "rect_short_nm", "rect_long_nm", "aspect", "rectangularity", "area_nm2"]
    else:
        cols = ["eq_diam_nm", "feret_max_nm", "feret_min_nm", "martin_nm", "aspect_ratio", "circularity", "solidity", "area_px", "mean_intensity"]
    rows = []
    for i, p in enumerate(result["particles"], 1):
        row = {"particle": i}
        row.update({c: p[c] for c in cols})
        row["centroid_row"], row["centroid_col"] = p["centroid"]
        rows.append(row)
    return pd.DataFrame(rows)


def show_results(result, cfg):
    """Metadata summary, warnings and, if something was counted, overlay + histogram; otherwise the segmentation mask."""
    print("File: " + result["path"])
    print("Detector: " + str(result["detector"]) + " | mode used: " + ("dark-field (bright particles)" if result["is_darkfield"] else "bright-field (dark particles)") +
          " | calibration: " + format(result["nm_per_px"], ".4f") + " nm/px | field of view: " + format(result["fov_nm"], ".0f") + " nm")
    print("Counted: " + str(len(result["particles"])) + " | cut by image border: " + str(len(result["border"])) +
          " | excluded (" + ("not a single cube" if cfg["shape"] == "cube" else "overlapping") + "): " + str(len(result["excluded"])) +
          ((" | oversized: " + str(len(result["oversized"]))) if cfg["shape"] != "cube" else ""))
    for w in result["warnings"]:
        print("WARNING: " + w)
    if not result["particles"]:
        plt.figure(figsize=(7, 7))
        plt.imshow(result["overlay"][:, :, ::-1] if result["overlay"] is not None else result["mask"], cmap="gray")
        plt.title("Regions found (none counted)")
        plt.axis("off")
        plt.show()
        return
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(result["overlay"], cv2.COLOR_BGR2RGB))
    legend = "green = counted, orange = not a single cube, cyan = cut by border" if cfg["shape"] == "cube" else "red = counted, orange = overlapping, cyan = cut by border, magenta = oversized"
    plt.title("Detected (N=" + str(result["stats"]["n"]) + ")\n" + legend)
    plt.axis("off")
    plt.show()
    plot_histogram([p[cfg["measure"]] for p in result["particles"]], result["stats"], cfg["measure"], material=cfg["material"], bins=cfg["hist_bins"])


def save_results(result, outdir, cfg):
    """Saves overlay.png, histogram.png and particles.csv of one result into outdir, as selected in the panel."""
    if not result["particles"]:
        return []
    os.makedirs(outdir, exist_ok=True)
    saved = []
    if cfg["save_overlay"]:
        cv2.imwrite(os.path.join(outdir, "overlay.png"), result["overlay"]); saved.append("overlay.png")
    if cfg["save_histogram"]:
        plot_histogram([p[cfg["measure"]] for p in result["particles"]], result["stats"], cfg["measure"], out_path=os.path.join(outdir, "histogram.png"),
                       material=cfg["material"], show=False, bins=cfg["hist_bins"])
        saved.append("histogram.png")
    if cfg["save_csv"]:
        particles_table(result).to_csv(os.path.join(outdir, "particles.csv"), index=False); saved.append("particles.csv")
    return saved


def summarize_results(results):
    """One row per file: calibration, counts, statistics of the chosen measure, notes."""
    rows = []
    for name, res in results.items():
        row = {"image": name, "nm_per_px": res["nm_per_px"], "n_cut_by_border": len(res["border"]), "n_excluded": len(res["excluded"]),
               "n_oversized": len(res["oversized"])}
        if res["stats"] is not None:
            row.update(res["stats"])
        else:
            row["n"] = 0
        row["notes"] = "; ".join(res["warnings"])
        rows.append(row)
    df = pd.DataFrame(rows).set_index("image").sort_index()
    return df.rename(columns={"mean_nm": "mean", "median_nm": "median", "std_nm": "std", "min_nm": "min", "max_nm": "max"})

## 3. The panel

`FriendlyPanel` builds the three steps. The defaults are in `DEFAULTS`.


In [8]:
DEFAULTS = {
    "shape": "sphere", "mode": "folder", "file_path": "./Data/Velox/STEM HAADF 1532.emd", "folder_path": "./Data/Velox", "preview_file": None,
    "contrast": "auto",
    "min_diam_nm": 5.0, "min_circularity": 0.5, "separate_touching": True, "min_peak_distance_nm": 5.0, "smooth_nm": 0.3,
    "exclude_border": True, "max_intensity_ratio": 1.3, "max_diameter_ratio": 2.5,
    "cube_min_rectangularity": 0.80, "cube_max_aspect": 1.5, "cube_min_side_px": 10,
    "measure": "eq_diam_nm", "hist_bins": 30, "material": "nanoparticles", "overlay_style": "circles",
    "outdir": "Result_Poly_emd", "save_results": True, "save_overlay": True, "save_histogram": True, "save_csv": True,
    "write_log": True, "show_plots": True,
}
PREVIEW_PX = 1024      # the live preview works on a copy of the image reduced to about this size
PREVIEW_KEYS = ("shape", "min_diam_nm", "min_circularity", "separate_touching", "min_peak_distance_nm", "exclude_border", "max_intensity_ratio",
                "max_diameter_ratio", "cube_min_rectangularity", "cube_max_aspect", "cube_min_side_px", "measure", "overlay_style", "contrast")

CSS = """
<style>
.dm4f { font-family: var(--jp-ui-font-family, system-ui, -apple-system, 'Segoe UI', sans-serif); color: var(--jp-ui-font-color1, #222); max-width: 1020px; }
.dm4f .title { font-size: 20px; font-weight: 600; margin: 2px 0 0; }
.dm4f .sub { color: var(--jp-ui-font-color2, #666); font-size: 13px; margin: 0 0 6px; }
.dm4f .stepbar { border-bottom: 1px solid var(--jp-border-color2, #ddd); margin: 6px 0 10px; }
.dm4f .jupyter-button.stepbtn { background: transparent !important; border: none !important; border-bottom: 3px solid transparent !important; border-radius: 0 !important;
  color: var(--jp-ui-font-color2, #666) !important; font-size: 14px; box-shadow: none !important; height: 38px; padding: 0 18px; }
.dm4f .jupyter-button.stepbtn.active { border-bottom-color: var(--jp-brand-color1, #1976d2) !important; color: var(--jp-ui-font-color1, #222) !important; font-weight: 600; }
.dm4f .jupyter-button.stepbtn.done { color: var(--jp-success-color1, #2e7d32) !important; }
.dm4f .card { background: var(--jp-layout-color1, #fff); border: 1px solid var(--jp-border-color2, #e0e0e0); border-radius: 12px; padding: 12px 16px; margin: 6px 0; }
.dm4f .gt { font-size: 12px; font-weight: 600; letter-spacing: .04em; text-transform: uppercase; color: var(--jp-ui-font-color2, #666); margin: 2px 0 4px; }
.dm4f .lab { font-size: 13px; padding-top: 6px; }
.dm4f .hint { font-size: 12px; color: var(--jp-ui-font-color2, #777); }
.dm4f .tiles { display: grid; grid-template-columns: repeat(auto-fit, minmax(92px, 1fr)); gap: 8px; margin: 4px 0; }
.dm4f .tile { background: var(--jp-layout-color2, #f4f4f5); border-radius: 10px; padding: 8px 12px; }
.dm4f .tile .k { font-size: 11px; color: var(--jp-ui-font-color2, #777); }
.dm4f .seg .jupyter-button:first-child { border-radius: 8px 0 0 8px !important; }
.dm4f .seg .jupyter-button:last-child { border-radius: 0 8px 8px 0 !important; }
.dm4f .tile .v { font-size: 16px; font-weight: 600; }
.dm4f .msg { border-radius: 8px; padding: 7px 12px; font-size: 13px; margin: 4px 0; }
.dm4f .msg.ok { background: rgba(46,125,50,.12); color: var(--jp-success-color1, #2e7d32); }
.dm4f .msg.warn { background: rgba(237,108,2,.13); color: var(--jp-warn-color1, #b45f06); }
.dm4f .msg.err { background: rgba(211,47,47,.12); color: var(--jp-error-color1, #c62828); }
.dm4f .seg { display: flex !important; flex-direction: row !important; flex-wrap: wrap; width: auto !important; }
.dm4f .seg .jupyter-button { width: auto !important; min-width: 64px; padding: 0 14px !important; margin: 0 !important; border-radius: 0 !important; background: var(--jp-layout-color2, #f4f4f5) !important; color: var(--jp-ui-font-color1, #222) !important; border: 1px solid var(--jp-border-color2, #ddd) !important;
  box-shadow: none !important; height: 30px; font-size: 13px; }
.dm4f .seg .jupyter-button.mod-active { background: var(--jp-brand-color1, #1976d2) !important; color: #fff !important; border-color: var(--jp-brand-color1, #1976d2) !important; }
.dm4f .jupyter-button.primary { background: var(--jp-brand-color1, #1976d2) !important; color: #fff !important; border: none !important; border-radius: 8px !important; font-weight: 600; height: 36px; }
.dm4f .jupyter-button.ghost { background: transparent !important; border: 1px solid var(--jp-border-color1, #bbb) !important; border-radius: 8px !important; height: 36px; color: var(--jp-ui-font-color1, #222) !important; }
.dm4f table.sum { border-collapse: collapse; font-size: 13px; width: 100%; }
.dm4f table.sum td { padding: 3px 8px 3px 0; border-bottom: 1px solid var(--jp-border-color3, #eee); }
.dm4f table.sum td.k { color: var(--jp-ui-font-color2, #777); width: 46%; }
</style>
"""


def _esc(t):
    return str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _tiles(pairs):
    return "<div class='tiles'>" + "".join("<div class='tile'><div class='k'>" + _esc(k) + "</div><div class='v'>" + _esc(v) + "</div></div>" for k, v in pairs) + "</div>"


class FriendlyPanel:
    STEPS = ["1  Data", "2  Settings", "3  Run"]

    def __init__(self, defaults=None):
        self.cfg = dict(DEFAULTS)
        if defaults:
            self.cfg.update(defaults)
        self.results, self.summary = {}, None
        self.step = 0
        self._files, self._header, self._cache = [], None, {}
        self._busy = False
        self._lock = False
        self._build()
        self._scan()
        self.go(0)

    # ------------------------------------------------------------------ helpers
    def _bind(self, key, w):
        w.observe(lambda ch: self._set(key, ch["new"]), "value")
        return w

    def _seg(self, key, options):
        w = W.ToggleButtons(options=options, value=self.cfg[key], layout=W.Layout(width="auto"))
        w.add_class("seg")
        return self._bind(key, w)

    def _slider(self, key, lo, hi, step, fmt, cls=W.FloatSlider):
        w = cls(value=self.cfg[key], min=lo, max=hi, step=step, continuous_update=False, layout=W.Layout(width="260px"))
        if fmt:
            w.readout_format = fmt
        return self._bind(key, w)

    def _row(self, label, widget, hint=None):
        lab = W.HTML("<div class='lab'>" + _esc(label) + "</div>", layout=W.Layout(width="190px", flex="0 0 190px"))
        row = W.HBox([lab, widget], layout=W.Layout(align_items="flex-start", margin="1px 0"))
        parts = [row]
        hint_w = None
        if hint:
            hint_w = W.HTML("<div class='hint' style='margin:-2px 0 4px 190px'>" + _esc(hint) + "</div>")
            parts.append(hint_w)
        box = W.VBox(parts)
        box.lab, box.hint_w = lab, hint_w
        return box

    def _relabel(self, row, label, hint=None):
        row.lab.value = "<div class='lab'>" + _esc(label) + "</div>"
        if hint is not None and row.hint_w is not None:
            row.hint_w.value = "<div class='hint' style='margin:-2px 0 4px 190px'>" + _esc(hint) + "</div>"

    # ------------------------------------------------------------------ layout
    def _build(self):
        c = self.cfg
        gt = lambda t: W.HTML("<div class='gt'>" + t + "</div>")
        self.head = W.HTML(CSS + "<div class='title'>Nanoparticle size analysis</div><div class='sub'>Thermo Fisher .emd files &middot; polydispersity &middot; round particles and cubic crystals</div>")
        self.step_btns = [W.Button(description=t, layout=W.Layout(width="auto")) for t in self.STEPS]
        for i, b in enumerate(self.step_btns):
            b.add_class("stepbtn")
            b.on_click(lambda _b, i=i: self.go(i))
        self.stepbar = W.HBox(self.step_btns)
        self.stepbar.add_class("stepbar")

        # ---- step 1: data and shape
        self.mode = self._seg("mode", [("Folder", "folder"), ("Single file", "file")])
        self.path = W.Text(value=c["folder_path"] if c["mode"] == "folder" else c["file_path"], continuous_update=False,
                           placeholder="Path to the data", layout=W.Layout(width="100%"))
        self.path.observe(lambda ch: self._on_path(ch["new"]), "value")
        self.b_browse = W.Button(description="Browse...", layout=W.Layout(width="110px"))
        self.b_browse.add_class("ghost")
        self.b_browse.on_click(self._toggle_browser)
        self.chooser_box = W.VBox([], layout=W.Layout(display="none"))
        self.shape = self._seg("shape", [("Round particles", "sphere"), ("Cubic crystals", "cube")])
        self.contrast = self._seg("contrast", [("Auto", "auto"), ("Dark particles", "dark"), ("Bright particles", "bright")])
        self.preview_file = W.Dropdown(options=[], layout=W.Layout(width="300px"))
        self.preview_file.observe(lambda ch: self._on_preview_file(ch["new"]), "value")
        self.row_preview_file = self._row("Preview image", self.preview_file)
        self.info = W.HTML("")
        self.card_data = W.VBox([gt("Data"), self._row("Analyze", self.mode), W.HBox([self.path, self.b_browse], layout=W.Layout(width="100%", margin="4px 0")), self.chooser_box])
        self.card_data.add_class("card")
        self.row_shape = self._row("Particle shape", self.shape, " ")
        self.card_info = W.VBox([gt("What was found"), self.info, self.row_preview_file, self.row_shape,
                                 self._row("Particle contrast", self.contrast, "Auto reads it from the detector: HAADF / DF = bright particles, BF = dark particles.")])
        self.card_info.add_class("card")
        self.page_data = W.VBox([self.card_data, self.card_info])

        # ---- step 2: settings + live preview
        self.w_min_diam = self._slider("min_diam_nm", 1.0, 100.0, 0.5, ".1f")
        self.row_min_diam = self._row("Minimum diameter (nm)", self.w_min_diam, "Smaller regions are noise. In nm: independent of the magnification.")
        self.w_circ = self._slider("min_circularity", 0.0, 1.0, 0.05, ".2f")
        self.row_circ = self._row("Minimum circularity", self.w_circ, "1 = perfect circle. Lower it for irregular particles.")
        self.w_split = self._seg("separate_touching", [("Yes", True), ("No", False)])
        self.row_split = self._row("Split touching particles", self.w_split)
        self.w_peak = self._slider("min_peak_distance_nm", 1.0, 100.0, 0.5, ".1f")
        self.row_peak = self._row("Min. centre distance (nm)", self.w_peak, "About the radius of the smallest particle.")
        self.w_border = self._seg("exclude_border", [("Yes", True), ("No", False)])
        self.row_border = self._row("Exclude cut particles", self.w_border, "Particles touching the image border are not counted.")
        self.w_overlap = self._seg("max_intensity_ratio", [("Off", 0.0), ("1.3x", 1.3), ("1.5x", 1.5), ("2x", 2.0)])
        self.row_overlap = self._row("Exclude overlapped", self.w_overlap, "Dark-field only: where particles overlap the signal adds up. Drops particles brighter than this multiple of the median.")
        self.w_over = self._seg("max_diameter_ratio", [("Off", 0.0), ("2x mean", 2.0), ("2.5x", 2.5), ("3x", 3.0)])
        self.row_over = self._row("Exclude oversized", self.w_over, "Drops merged clusters (diameter above this multiple of the mean).")
        self.w_rect = self._slider("cube_min_rectangularity", 0.5, 1.0, 0.01, ".2f")
        self.row_rect = self._row("Minimum rectangularity", self.w_rect, "Area / bounding-rectangle area. Lower it for cubes with rounded corners.")
        self.w_aspect = self._slider("cube_max_aspect", 1.0, 3.0, 0.05, ".2f")
        self.row_aspect = self._row("Maximum aspect ratio", self.w_aspect, "Long / short side of the rectangle. Drops elongated regions.")
        self.w_sidepx = self._slider("cube_min_side_px", 4, 60, 1, "d", cls=W.IntSlider)
        self.row_sidepx = self._row("Minimum side (pixels)", self.w_sidepx, "A few pixels are always 'rectangular' and their side is only known to the pixel size.")
        self.card_detect = W.VBox([gt("Detection"), self.row_min_diam, self.row_circ, self.row_split, self.row_peak, self.row_rect, self.row_aspect, self.row_sidepx,
                                   self.row_border, self.row_overlap, self.row_over])
        self.card_detect.add_class("card")
        self.w_measure = W.Dropdown(options=[], layout=W.Layout(width="270px"))
        self._set_measure_options()
        self._bind("measure", self.w_measure)
        self.w_style = self._seg("overlay_style", [("Circles", "circles"), ("Contours", "contours")])
        self.row_style = self._row("Overlay style", self.w_style)
        self.w_bins = W.BoundedIntText(value=c["hist_bins"], min=3, max=200, layout=W.Layout(width="80px"))
        self._bind("hist_bins", self.w_bins)
        self.card_measure = W.VBox([gt("Measure"), self._row("Size measure", self.w_measure, "Used for the statistics and the histogram. The CSV always has all the measures."),
                                    self._row("Histogram bins", self.w_bins), self.row_style])
        self.card_measure.add_class("card")
        self.prev_status = W.HTML("")
        self.prev_img = W.Image(value=b"", format="png", layout=W.Layout(width="100%", max_width="520px"))
        self.prev_tiles = W.HTML("")
        self.prev_msgs = W.HTML("")
        self.b_prev = W.Button(description="Refresh preview", layout=W.Layout(width="150px"))
        self.b_prev.add_class("ghost")
        self.b_prev.on_click(lambda _b: self._preview(force=True))
        self.card_prev = W.VBox([gt("Preview on one image"), self.prev_status, self.prev_img, self.prev_tiles, self.prev_msgs, self.b_prev],
                                layout=W.Layout(flex="1 1 46%", min_width="320px"))
        self.card_prev.add_class("card")
        left = W.VBox([self.card_detect, self.card_measure], layout=W.Layout(flex="1 1 52%", min_width="360px"))
        self.page_set = W.HBox([left, self.card_prev], layout=W.Layout(flex_flow="row wrap", align_items="flex-start", width="100%"))

        # ---- step 3: run
        self.w_save = self._seg("save_results", [("Yes", True), ("No", False)])
        self.w_outdir = W.Text(value=c["outdir"], continuous_update=False, layout=W.Layout(width="320px"))
        self._bind("outdir", self.w_outdir)
        cb = lambda key, text: self._bind(key, W.Checkbox(value=c[key], description=text, indent=False, layout=W.Layout(width="auto", margin="0 14px 0 0")))
        self.w_saveopts = W.HBox([cb("save_overlay", "Overlay"), cb("save_histogram", "Histogram"), cb("save_csv", "CSV tables"), cb("write_log", "Run log")],
                                 layout=W.Layout(flex_flow="row wrap"))
        self.w_show = self._seg("show_plots", [("All figures", True), ("Summary only", False)])
        self.row_outdir = self._row("Output folder", self.w_outdir, "Single file: results directly here. Folder: one subfolder per image plus summary.csv, combined histogram and run_log.txt.")
        self.row_saveopts = self._row("Save", self.w_saveopts)
        self.card_out = W.VBox([gt("Results"), self._row("Save to disk", self.w_save), self.row_outdir, self.row_saveopts, self._row("Show in notebook", self.w_show)])
        self.card_out.add_class("card")
        self.summary_html = W.HTML("")
        self.card_sum = W.VBox([gt("Summary of the settings"), self.summary_html])
        self.card_sum.add_class("card")
        self.run_msg = W.HTML("")
        self.b_run = W.Button(description="Run analysis", layout=W.Layout(width="190px"))
        self.b_run.add_class("primary")
        self.b_run.on_click(self._on_run)
        self.page_run = W.VBox([self.card_out, self.card_sum, self.run_msg, self.b_run])

        self.b_back = W.Button(description="Back", layout=W.Layout(width="110px"))
        self.b_back.add_class("ghost")
        self.b_back.on_click(lambda _b: self.go(self.step - 1))
        self.b_next = W.Button(description="Next", layout=W.Layout(width="110px"))
        self.b_next.add_class("primary")
        self.b_next.on_click(lambda _b: self.go(self.step + 1))
        self.nav = W.HBox([self.b_back, self.b_next], layout=W.Layout(justify_content="space-between", margin="8px 0"))
        self.pages = [self.page_data, self.page_set, self.page_run]
        self.out = W.Output()
        self.root = W.VBox([self.head, self.stepbar, W.VBox(self.pages), self.nav])
        self.root.add_class("dm4f")
        self.out_box = W.VBox([self.out])
        self.out_box.add_class("dm4f")

    def _set_measure_options(self):
        table = CUBE_MEASURES if self.cfg["shape"] == "cube" else SPHERE_MEASURES
        self._lock = True
        self.w_measure.options = [(measure_label(k), k) for k in table]
        if self.cfg["measure"] not in table:
            self.cfg["measure"] = DEFAULT_MEASURE[self.cfg["shape"]]
        self.w_measure.value = self.cfg["measure"]
        self._lock = False

    # ------------------------------------------------------------------ state changes
    def _on_path(self, value):
        key = "folder_path" if self.cfg["mode"] == "folder" else "file_path"
        self.cfg[key] = str(value).strip().strip('"')
        self.cfg["preview_file"] = None
        self._scan()

    def _on_preview_file(self, value):
        if value and not self._lock:
            self.cfg["preview_file"] = value
            self._scan()

    def _set(self, key, value):
        if self._lock:
            return
        self.cfg[key] = value
        if key == "mode":
            self.path.value = self.cfg["folder_path"] if value == "folder" else self.cfg["file_path"]
            self.cfg["preview_file"] = None
            self._scan()
        if key == "shape":
            self._set_measure_options()
        if key in ("contrast", "shape"):
            self._scan()
        self._visibility()
        if self.step == 1 and key in PREVIEW_KEYS:
            self._preview()
        if self.step == 2:
            self._render_summary()

    def _visibility(self):
        c = self.cfg
        cube = c["shape"] == "cube"
        show = lambda w, on: setattr(w.layout, "display", "" if on else "none")
        for row in (self.row_circ, self.row_split, self.row_overlap, self.row_over, self.row_style):
            show(row, not cube)
        show(self.row_peak, (not cube) and c["separate_touching"])
        for row in (self.row_rect, self.row_aspect, self.row_sidepx):
            show(row, cube)
        self._relabel(self.row_min_diam, "Minimum side (nm)" if cube else "Minimum diameter (nm)")
        self._relabel(self.row_border, "Exclude cut cubes" if cube else "Exclude cut particles")
        show(self.row_outdir, c["save_results"])
        show(self.row_saveopts, c["save_results"])
        show(self.row_preview_file, len(self._files) > 1)
        self._relabel(self.row_shape, "Particle shape",
                      "Cubic crystals: finds the dark lines between touching cubes and measures the side. HAADF (dark-field) images only; stacked cubes are not counted." if cube else
                      "Round particles: Otsu threshold and watershed. Cubic crystals: a dedicated pipeline that measures the side of each cube.")

    def _toggle_browser(self, _b):
        if self.chooser_box.children:
            self.chooser_box.children = []
            self.chooser_box.layout.display = "none"
            return
        folder = self.cfg["mode"] == "folder"
        cur = self.cfg["folder_path"] if folder else self.cfg["file_path"]
        parent = os.path.dirname(os.path.abspath(cur))
        start = cur if os.path.isdir(cur) else (parent if os.path.exists(parent) else os.path.abspath("."))
        fc = FileChooser(os.path.abspath(start), show_only_dirs=folder, select_desc="Use this", change_desc="Change",
                         filter_pattern=None if folder else ["*.emd"], title="Pick the folder" if folder else "Pick the file")
        fc.layout.width = "100%"

        def chosen(_c):
            if fc.selected:
                self.path.value = fc.selected
                self.chooser_box.children = []
                self.chooser_box.layout.display = "none"
        fc.register_callback(chosen)
        self.chooser_box.children = [fc]
        self.chooser_box.layout.display = ""

    # ------------------------------------------------------------------ scanning the data
    def _scan(self):
        c = self.cfg
        if c["mode"] == "file":
            self._files = [c["file_path"]] if os.path.isfile(c["file_path"]) else []
        else:
            self._files = list_emd_files(c["folder_path"]) if os.path.isdir(c["folder_path"]) else []
        self._header = None
        msgs = []
        if not self._files:
            what = c["file_path"] if c["mode"] == "file" else c["folder_path"]
            self.info.value = "<div class='msg err'>No .emd data found at: " + _esc(what) + "</div>"
            self._lock = True
            self.preview_file.options = []
            self._lock = False
            self._visibility()
            return
        self._lock = True      # changing the options must not trigger a new scan
        self.preview_file.options = [(os.path.basename(f), f) for f in self._files]
        if c["preview_file"] in self._files:
            self.preview_file.value = c["preview_file"]
        else:
            self.preview_file.value = self._files[0]
            c["preview_file"] = self._files[0]
        self._lock = False
        f = c["preview_file"]
        try:
            h = read_emd_header(f)
            self._header = h
            guess = "Dark-field (bright particles)" if h["is_darkfield"] else "Bright-field (dark particles)"
            used = {"auto": guess, "dark": "Dark particles (forced)", "bright": "Bright particles (forced)"}[c["contrast"]]
            html = _tiles([("Files", str(len(self._files))), ("Scale", format(h["nm_per_px"], ".3f") + " nm/px"), ("Field of view", format(h["fov_nm"], ".0f") + " nm"),
                           ("Signal", h["signal"] + (" (1 of " + str(h["n_signals"]) + ")" if h["n_signals"] > 1 else "")), ("Detector", str(h["detector"])), ("Contrast", used)])
            if h["fov_nm"] < 100:
                msgs.append(("warn", "The field of view is only " + format(h["fov_nm"], ".0f") + " nm: this looks like atomic resolution. Whole particles will not fit; it is not suited for counting."))
            elif h["nm_per_px"] > 3:
                msgs.append(("warn", "Low resolution (" + format(h["nm_per_px"], ".1f") + " nm/px): small particles will be only a few pixels wide."))
            else:
                msgs.append(("ok", "Calibration read from the metadata. The scale and the field of view are in the usual range for counting."))
            if c["shape"] == "cube" and not h["is_darkfield"] and c["contrast"] != "bright":
                msgs.append(("warn", "Cubic crystals need a dark-field (HAADF) image: this signal looks like bright-field."))
            self.info.value = html + "".join("<div class='msg " + k + "'>" + _esc(t) + "</div>" for k, t in msgs)
        except Exception as exc:
            self.info.value = "<div class='msg err'>Cannot read the metadata of " + _esc(os.path.basename(f)) + ": " + _esc(exc) + "</div>"
        self._visibility()

    # ------------------------------------------------------------------ steps
    def go(self, n):
        n = max(0, min(2, n))
        if n > 0 and not self._files:
            n = 0
        self.step = n
        for i, (p, b) in enumerate(zip(self.pages, self.step_btns)):
            p.layout.display = "" if i == n else "none"
            for cls in ("active", "done"):
                b.remove_class(cls)
            if i == n:
                b.add_class("active")
            elif i < n:
                b.add_class("done")
        self.b_back.layout.visibility = "hidden" if n == 0 else "visible"
        self.b_next.layout.display = "none" if n == 2 else ""
        self._visibility()
        if n == 1:
            self._preview()
        if n == 2:
            self._render_summary()

    def _render_summary(self):
        c = self.cfg
        n = len(self._files)
        cube = c["shape"] == "cube"
        if cube:
            det = [("Minimum side / pixels", format(c["min_diam_nm"], "g") + " nm / " + str(c["cube_min_side_px"]) + " px"),
                   ("Rectangularity / aspect", ">= " + format(c["cube_min_rectangularity"], "g") + " / <= " + format(c["cube_max_aspect"], "g")),
                   ("Exclude cut cubes", "Yes" if c["exclude_border"] else "No")]
        else:
            det = [("Minimum diameter / circularity", format(c["min_diam_nm"], "g") + " nm / " + format(c["min_circularity"], "g")),
                   ("Split touching particles", ("Yes, min. centre distance " + format(c["min_peak_distance_nm"], "g") + " nm") if c["separate_touching"] else "No"),
                   ("Exclude cut / overlapped / oversized", ("Yes" if c["exclude_border"] else "No") + " / " + ("off" if not c["max_intensity_ratio"] else format(c["max_intensity_ratio"], "g") + "x median") +
                    " / " + ("off" if not c["max_diameter_ratio"] else format(c["max_diameter_ratio"], "g") + "x mean"))]
        rows = [("Data", ("Folder: " if c["mode"] == "folder" else "File: ") + _esc(c["folder_path"] if c["mode"] == "folder" else c["file_path"]) + " (" + str(n) + " file" + ("s" if n != 1 else "") + ")"),
                ("Particle shape", "Cubic crystals" if cube else "Round particles"),
                ("Particle contrast", {"auto": "Auto (metadata)", "dark": "Dark particles", "bright": "Bright particles"}[c["contrast"]])] + det + [
                ("Size measure", measure_label(c["measure"])),
                ("Output", _esc(os.path.abspath(c["outdir"])) if c["save_results"] else "Nothing is saved")]
        self.summary_html.value = "<table class='sum'>" + "".join("<tr><td class='k'>" + k + "</td><td>" + v + "</td></tr>" for k, v in rows) + "</table>"
        msg = ""
        if not self._files:
            msg = "<div class='msg err'>No data selected: go back to step 1.</div>"
        elif c["save_results"] and not (c["save_overlay"] or c["save_histogram"] or c["save_csv"] or c["write_log"]):
            msg = "<div class='msg warn'>Saving is on but every output type is off: nothing will be written.</div>"
        self.run_msg.value = msg

    # ------------------------------------------------------------------ live preview
    def _load_small(self, path):
        if path not in self._cache:
            gray, nm, df, info = load_emd_image(path)
            f = max(1, int(np.ceil(max(gray.shape) / PREVIEW_PX)))
            info = dict(info)
            if f > 1:
                size = (gray.shape[1] // f, gray.shape[0] // f)
                gray = cv2.resize(gray, size, interpolation=cv2.INTER_AREA)
                info["raw"] = cv2.resize(info["raw"], size, interpolation=cv2.INTER_AREA)
            self._cache[path] = (gray, nm * f, df, info, f)
            if len(self._cache) > 4:
                self._cache.pop(next(iter(self._cache)))
        return self._cache[path]

    def _preview(self, force=False):
        if self._busy or not self._files:
            return
        path = self.cfg["preview_file"] or self._files[0]
        self._busy = True
        self.prev_status.value = "<div class='hint'>Computing...</div>"
        try:
            gray, nm_small, df, info, f = self._load_small(path)
            res = analyze_image(path, self.cfg, loaded=(gray, nm_small, df, info), with_martin=(self.cfg["measure"] == "martin_nm"))
            img = res["overlay"] if res["overlay"] is not None else cv2.cvtColor((res["mask"] * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
            self.prev_img.value = cv2.imencode(".png", img)[1].tobytes()
            st, unit = res["stats"], MEASURES[self.cfg["measure"]][1]
            tiles = [("Counted", str(st["n"])), ("Mean", format(st["mean_nm"], ".2f") + (" " + unit if unit else "")), ("CV", format(st["cv_percent"], ".1f") + " %"),
                     ("PDI", format(st["pdi"], ".3f"))] if st else [("Counted", "0")]
            self.prev_tiles.value = _tiles(tiles)
            self.prev_msgs.value = "".join("<div class='msg warn'>" + _esc(w) + "</div>" for w in res["warnings"])
            note = "Preview on " + os.path.basename(path) + (" (reduced " + str(f) + "x for speed; the final run uses the full resolution)" if f > 1 else "")
            self.prev_status.value = "<div class='hint'>" + _esc(note) + "</div>"
        except Exception as exc:
            self.prev_status.value = "<div class='msg err'>" + _esc(type(exc).__name__ + ": " + str(exc)) + "</div>"
        finally:
            self._busy = False

    # ------------------------------------------------------------------ run
    def _on_run(self, _b):
        with self.out:
            clear_output(wait=True)
            if not self._files:
                print("No data selected."); return
            self.b_run.disabled = True
            try:
                self._run_all()
            except Exception as exc:
                print("ERROR: " + type(exc).__name__ + ": " + str(exc))
            finally:
                self.b_run.disabled = False

    def _run_all(self):
        c = dict(self.cfg)
        files = self._files
        folder_mode = c["mode"] == "folder"
        save, base = c["save_results"], c["outdir"]
        log = ["GAIN Polydispersity - EMD Vol.02 Friendly run log", "Date: " + datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
               "Mode: " + c["mode"] + " | files: " + str(len(files)), "", "Settings:"] + ["  " + k + " = " + str(v) for k, v in c.items()] + [""]
        bar = W.IntProgress(min=0, max=len(files), description="Files:", layout=W.Layout(width="60%"))
        display(bar)
        results = {}
        for path in files:
            name = os.path.splitext(os.path.basename(path))[0]
            try:
                res = analyze_image(path, c)
            except Exception as exc:
                print("[SKIPPED] " + name + ": " + str(exc)); log.append("[SKIPPED] " + name + ": " + str(exc)); bar.value += 1
                continue
            results[name] = res
            s = res["stats"]
            line = name + ": " + ("N=" + str(s["n"]) + ", mean=" + format(s["mean_nm"], ".2f") + ", CV=" + format(s["cv_percent"], ".1f") + "%, PDI=" + format(s["pdi"], ".4f") if s else "nothing counted")
            print(line)
            log.append(line + " | cut by border: " + str(len(res["border"])) + " | excluded: " + str(len(res["excluded"])) + " | oversized: " + str(len(res["oversized"])))
            for w in res["warnings"]:
                print("   WARNING: " + w); log.append("   WARNING: " + w)
            if c["show_plots"]:
                show_results(res, c)
            if save:
                save_results(res, os.path.join(base, name) if folder_mode else base, c)
            bar.value += 1
        self.results = results
        if not results:
            print("No file could be analyzed."); return
        self.summary = summarize_results(results)
        print("\nSummary (" + measure_label(c["measure"]) + "):")
        display(self.summary)
        if folder_mode:
            groups = [[p[c["measure"]] for p in r["particles"]] for r in results.values() if r["particles"]]
            if groups:
                pooled = np.concatenate(groups)
                pstats = polydispersity_stats(pooled)
                print("Combined distribution (all files pooled):")
                plot_histogram(pooled, pstats, c["measure"], material=c["material"], bins=c["hist_bins"],
                               out_path=os.path.join(base, "combined_histogram.png") if save and c["save_histogram"] else None)
                display(pd.Series(pstats).to_frame().T)
                log.append("Pooled: " + str(pstats))
                if save and c["save_csv"]:
                    allp = pd.concat([particles_table(r).assign(image=n) for n, r in results.items() if r["particles"]], ignore_index=True)
                    os.makedirs(base, exist_ok=True)
                    allp.to_csv(os.path.join(base, "combined_particles.csv"), index=False)
            if save and c["save_csv"]:
                os.makedirs(base, exist_ok=True)
                self.summary.to_csv(os.path.join(base, "summary.csv"))
        if save and c["write_log"]:
            os.makedirs(base, exist_ok=True)
            with open(os.path.join(base, "run_log.txt"), "w", encoding="utf-8") as f:
                f.write("\n".join(log) + "\n")
        if save:
            print("\nResults saved in: " + os.path.abspath(base))

    def show(self):
        display(self.root, self.out_box)
        return self


def launch(defaults=None):
    """Opens the panel. Returns it: panel.cfg = settings, panel.results / panel.summary = last run."""
    return FriendlyPanel(defaults).show()

## 4. Launch

Run this cell to open the panel. After **Run analysis** the results stay available in Python: `panel.results` (one entry per image), `panel.summary` (table), `panel.cfg` (the settings used).


In [9]:
panel = launch()